# Building a Telecom Revenue-Assurance Summary Cube with PROC SUMMARY

## Executive Summary

A revenue-assurance team at a wireless carrier pre-aggregates a month of subscriber-day billing records into a compact summary cube so analysts can drill down on settled revenue across region, plan tier, and call type without re-scanning the raw fact table. `PROC SUMMARY` rolls 100 subscriber-day records into a multidimensional set of cells: the finest grain (region x plan tier x call type) fills 25 of 27 possible cells, while named subcubes give the marginals analysts query most. In this sample month the carrier settled \$222.78 across the three active regions, with South (\$97.44) and East (\$86.94) together accounting for 83% of revenue and North trailing at \$38.40. The richest single subcube is the Plus-tier Voice service (\$59.06 over 18 records), and ranking cells by per-record yield surfaces data-usage cells as the highest-value targets for a leakage audit (top yield \$7.87/record). Every figure below is read directly from the executed output.

## Data Sources

| Dataset | Rows | Grain | Key Variables |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | One row per subscriber-day usage summary | `region` (East/South/North), `plan_tier` (Prepaid/Basic/Plus), `call_type` (Voice/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | One row per non-empty (region x plan_tier x call_type) cell | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | One row per cell of the named drill subcubes | `_TYPE_`, `_FREQ_`, `rev_total` |

All data is generated inline with `call streaminit()` / `rand()`; no external files or network access are required. The fixed `streaminit(20260131)` seed makes the 100-record sample month fully reproducible, and the dimension probabilities are chosen so the finest-grain cube is densely populated (25 of the 27 possible cells filled).

## From raw call-detail records to a drillable cube

Wireless carriers settle revenue across millions of daily usage events. To let revenue-assurance analysts answer questions like *"What was Plus-tier voice revenue in the South region last month?"* without re-scanning the raw fact table every time, we **pre-aggregate** the data into a compact summary cube.

`PROC SUMMARY` is Base SAS's workhorse for exactly this: it groups a flat fact table by one or more `CLASS` dimensions and writes the requested statistics to an output dataset, tagging each row with `_TYPE_` (which dimensions are active) and `_FREQ_` (records behind the cell). That output dataset *is* a multidimensional cube — the same roll-up structure an OLAP tool would expose, materialized as an ordinary SAS dataset you can print, join, and slice.

This notebook:

1. Generates a realistic month of subscriber-day billing records.
2. Builds the finest-grain cube (region x plan tier x call type) with `PROC SUMMARY NWAY`.
3. Materializes named **drill subcubes** with the `TYPES` statement.
4. Projects revenue to the subscriber base with a `WEIGHT`, drills into one region, and ranks cells by per-record yield for leakage triage.

## Step 1 - Generate synthetic call-detail / billing data

Each row summarizes one subscriber's usage of one service on one day. We use `call streaminit` for reproducibility and `rand()` to draw plausible distributions: revenue and usage scale with the plan tier, voice revenue tracks billable minutes, and data revenue tracks megabytes. Each `RAND('table', ...)` is given one probability per category so every region, tier, and call type appears in the 100-record sample. A small `subscriber_wt` survey weight is attached so we can demonstrate a weighted measure later.

In [1]:
data work.cdr_billing;
    call streaminit(20260131);
    length region $6 plan_tier $9 call_type $7 device_class $8 bill_month $7;
    label revenue       = "Settled Revenue (USD)"
          call_minutes  = "Billable Voice Minutes"
          data_mb       = "Data Volume (MB)"
          subscriber_wt = "Subscriber Survey Weight";

    do i = 1 to 100;
        /* --- Dimensions: one probability per level, summing to 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        select (r);
            when (1) region = "East";
            when (2) region = "South";
            otherwise region = "North";
        end;

        p = rand("table", 0.30, 0.40, 0.30);
        select (p);
            when (1) plan_tier = "Prepaid";
            when (2) plan_tier = "Basic";
            otherwise plan_tier = "Plus";
        end;

        c = rand("table", 0.50, 0.30, 0.20);
        select (c);
            when (1) call_type = "Voice";
            when (2) call_type = "SMS";
            otherwise call_type = "Data";
        end;

        if rand("uniform") < 0.55 then device_class = "Smart";
        else device_class = "Feature";

        bill_month = "2026-01";

        /* --- Measures, scaled by tier and service --- */
        tier_mult = (plan_tier = "Prepaid")*0.7
                  + (plan_tier = "Basic")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Voice")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        output;
    end;
    drop i r p c tier_mult base_rev;
run;

proc print data=work.cdr_billing(obs=8) label noobs;
    title "Sample Subscriber-Day Billing Records";
run;

                                         Sample Subscriber-Day Billing Records                                          

region  plan_tier  call_type  device_class  bill_month  Billable Voice Minutes  Data Volume (MB)  Settled Revenue (USD)  Subscriber Survey Weight
North   Plus       SMS        Smart         2026-01                          0                 0                   0.67                      1.13
South   Prepaid    SMS        Feature       2026-01                          0                 0                   0.94                      1.42
South   Plus       SMS        Smart         2026-01                          0                 0                   0.99                      0.86
South   Plus       SMS        Smart         2026-01                          0                 0                   1.01                      1.03
South   Plus       Voice      Smart         2026-01                      103.4                 0                   5.79                      1.06
No

NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Step 2 - Build the finest-grain cube with PROC SUMMARY NWAY

`NWAY` keeps only the single most-detailed combination of all `CLASS` dimensions - here every populated (region x plan_tier x call_type) cell. For each cell we store the revenue `SUM`, `MEAN`, and `MAX` plus the total minutes and megabytes, so an analyst can read total revenue per cell, derive the per-record average, and spot the largest single charge. `_FREQ_` records how many subscriber-days back each cell. We drop `_TYPE_` here because, at the NWAY grain, every row has the same type.

In [2]:
proc summary data=work.cdr_billing nway;
    class region plan_tier call_type;
    var revenue call_minutes data_mb;
    output out=work.cube_nway(drop=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  max(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
run;

proc print data=work.cube_nway(obs=12) noobs;
    title "NWAY cube cells (region * plan_tier * call_type)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
run;

                                    NWAY cube cells (region * plan_tier * call_type)                                    

REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN  REV_MAX  MIN_TOTAL  DATA_TOTAL
East    Basic      Data            4      18.17      4.54     8.05       0.00    1,807.90
East    Basic      SMS             4       4.07      1.02     1.24       0.00        0.00
East    Basic      Voice           7      15.08      2.15     3.73     302.50        0.00
East    Plus       Data            1       5.54      5.54     5.54       0.00      573.90
East    Plus       SMS             4       3.59      0.90     0.95       0.00        0.00
East    Plus       Voice           7      23.87      3.41     8.01     491.70        0.00
East    Prepaid    SMS             3       3.00      1.00     1.06       0.00        0.00
East    Prepaid    Voice           8      13.62      1.70     6.50     270.20        0.00
North   Basic      Data            1       7.87      7.87     7.87  

NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Step 3 - Materialize named drill subcubes with TYPES

An OLAP cube pre-stores the roll-ups analysts navigate most. The `TYPES` statement does exactly that: each term asks `PROC SUMMARY` to emit one subcube. We request the grand total `()`, the `region` marginal, and the two-way `region*plan_tier` and `call_type*plan_tier` subcubes - the drill paths a revenue dashboard would expose. SAS tags each subcube with a `_TYPE_` code (a bitmask over the `CLASS` list), so a single output dataset carries every level of the hierarchy.

In [3]:
proc summary data=work.cdr_billing;
    class region plan_tier call_type;
    var revenue;
    types () region region*plan_tier call_type*plan_tier;
    output out=work.cube_hier
           sum(revenue)=rev_total;
run;

proc print data=work.cube_hier noobs;
    title "Subcube roll-ups: grand total, region, region*tier, calltype*tier";
    var _type_ region plan_tier call_type _freq_ rev_total;
    format rev_total comma10.2;
run;

                           Subcube roll-ups: grand total, region, region*tier, calltype*tier                            

_TYPE_  REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL
     0                                   100     222.78
     3          Basic      Data            8      38.06
     3          Basic      SMS             8       8.03
     3          Basic      Voice          20      42.33
     3          Plus       Data            3      17.46
     3          Plus       SMS            13      11.75
     3          Plus       Voice          18      59.06
     3          Prepaid    Data            7      14.50
     3          Prepaid    SMS             7       6.82
     3          Prepaid    Voice          16      24.77
     4  East                              38      86.94
     4  North                             23      38.40
     4  South                             39      97.44
     6  East    Basic                     15      37.32
     6  East    Plus                  

NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Step 4 - Weighted projection, a regional drill, and leakage triage

Three reads a revenue-assurance team actually performs against the cube:

- **Weighted projection.** Attaching `WEIGHT=subscriber_wt` to a `region*plan_tier` summary reports revenue scaled to the full subscriber base the sample represents, rather than the raw sampled rows.
- **Drill.** Filtering the NWAY cube to one region expands a single branch of the hierarchy - here South - to its tier-by-service detail.
- **Leakage triage.** Sorting the cells by mean revenue per record surfaces the highest-yield cells; thin-frequency, high-yield cells are exactly what an audit scrutinizes for mis-rated or leaking revenue.

In [4]:
/* Weighted revenue projected to the subscriber base */
proc summary data=work.cdr_billing nway;
    class region plan_tier;
    var revenue;
    weight subscriber_wt;
    output out=work.cube_wt(drop=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
run;

proc print data=work.cube_wt noobs;
    title "Weighted revenue by region * plan tier (projected to subscriber base)";
    format rev_weighted comma10.2;
run;

/* Drill into the South region branch of the cube */
proc print data=work.cube_nway noobs;
    where region = "South";
    title "Drill into South: revenue cells by tier and call type";
    var plan_tier call_type _freq_ rev_total rev_mean;
    format rev_total rev_mean comma10.2;
run;

/* Rank cells by per-record yield for leakage triage */
proc sort data=work.cube_nway out=work.cube_ranked;
    by descending rev_mean;
run;

proc print data=work.cube_ranked(obs=6) noobs;
    title "Highest average-revenue cells (per-record yield)";
    var region plan_tier call_type _freq_ rev_mean rev_max;
    format rev_mean rev_max comma10.2;
run;

                         Weighted revenue by region * plan tier (projected to subscriber base)                          

REGION  PLAN_TIER  REV_WEIGHTED  RECORDS
East    Basic             50.85       15
East    Plus              59.59       12
East    Prepaid           29.77       11
North   Basic             18.30        7
North   Plus              22.42        7
North   Prepaid           20.56        9
South   Basic             58.63       14
South   Plus              56.29       15
South   Prepaid           27.77       10

                                 Drill into South: revenue cells by tier and call type                                  

PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN
Basic      Data            3      12.02      4.01
Basic      SMS             2       2.01      1.00
Basic      Voice           9      22.51      2.50
Plus       Data            2      11.92      5.96
Plus       SMS             5       5.16      1.03
Plus       Voice           8      27.07      

NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpreting the results

The summary cube turns 100 raw subscriber-day rows into a compact set of pre-aggregated cells that answer drill-down questions instantly, without rescanning the fact table:

- **Where revenue lives.** The `region` marginal (`_TYPE_=4`) shows South settled \$97.44 and East \$86.94 - together 83% of the \$222.78 month - while North contributed \$38.40. Within the `call_type*plan_tier` subcube (`_TYPE_=3`), Plus-tier Voice is the single richest cell at \$59.06 over 18 records, with Basic-tier Voice next at \$42.33.
- **Weighted projection.** Applying the survey weight reshuffles the ranking toward plans whose subscribers carry more weight: East-Plus (\$59.59) and South-Basic (\$58.63) lead the projected `region*plan_tier` revenue, a different picture from the unweighted region totals and a reminder to report projected rather than sampled revenue when sizing exposure.
- **Per-record yield and leakage triage.** Ranking NWAY cells by `rev_mean` puts data-usage cells on top - North-Basic-Data (\$7.87/record) and South-Plus-Data (\$5.96/record) - confirming that data-heavy usage drives the highest per-record revenue. The thin-frequency leaders (one or two records) are precisely the cells a revenue-assurance analyst would pull the underlying call-detail records for, to confirm the high charge is correctly rated rather than an error.

For a revenue-assurance team, this cube is the foundation for variance dashboards: compare settled revenue against expected rate-card revenue per (region, tier, call type) cell, and the cells whose mean or weighted total deviate most become the leakage cases worth auditing. Because the whole structure is an ordinary SAS dataset, the next month's cube can be unioned, differenced, or joined to a rate card with the same Base SAS tools.